In [1]:
%run ./2_models.ipynb
%run ./3_tools.ipynb
%run ./4_utils.ipynb

C:\Users\Unreon\AppData\Local\Temp\ipykernel_36364\3082334981.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
USER_AGENT environment variable not set, consider setting it to identify your requests.


# LangGraph 기반 멀티 에이전트 시스템

LangGraph와 LangChain을 활용하여 여러 AI 에이전트가 협력해 사용자의 요구사항에 맞는 책의 목차(Outline)를 작성하는 시스템입니다.  
Supervisor가 작업을 계획하고 적절한 에이전트에 할당하며, 웹 검색과 벡터 검색을 통해 참고 자료를 수집합니다.  

## 주요 함수
| 함수 | 핵심 역할 | 주요 결과 |
|---|---|---|
| `supervisor()` | 다음 작업과 담당 에이전트 결정 | 새로운 `Task` 생성 |
| `supervisor_router()` | 다음 실행 노드 선택 | 에이전트 이름 반환 |
| `vector_search_agent()` | 벡터 DB 검색 및 문서 수집 | `references` 갱신 |
| `content_strategist()` | 책의 목차 생성 및 수정 | `outline.md` 저장 |
| `web_search_agent()` | 웹 검색 및 외부 자료 수집 | Chroma DB에 데이터 추가 |
| `communicator()` | 진행 상황 보고 및 사용자 응답 생성 | `messages` 갱신 |

In [2]:
import os 

current_path = os.getcwd()
filename = "multi_agent_tutorial.ipynb"
absolute_path = os.path.join(current_path, filename)

In [3]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-6-luna") # 모델 초기화

In [4]:
from langchain_core.messages import AnyMessage
from typing_extensions import TypedDict
from typing import List
#from models import Task
# %run ./models.ipynb로 불러온 Task 사용

# 상태 정의
class State(TypedDict):
    messages: List[AnyMessage | str]
    task_history: List[Task]    
    references: dict

## supervisor() — AI 에이전트 작업 관리 및 할당
AI 에이전트 팀의 작업을 관리하고, 현재 상황에서 수행해야 할 다음 작업을 결정하는 함수입니다.  
기존 대화 내용과 작성된 목차를 LLM에 전달하여 적절한 에이전트와 작업 내용을 결정하고, 이를 `Task` 객체로 생성하여 작업 이력에 추가합니다.

**매개변수**

| 매개변수 | 타입 | 설명 |
|---|---|---|
| `state` | `State` | 에이전트의 현재 실행 상태를 담은 딕셔너리 |

**State 주요 필드**

| 필드 | 설명 |
|---|---|
| `messages` | 사용자 및 AI 에이전트 간 대화 메시지 |
| `task_history` | 이전에 생성된 작업 이력 |

**반환값**

- `dict`: 갱신된 `messages`와 `task_history`를 반환합니다.
- 반환된 데이터는 LangGraph의 공유 상태에 반영됩니다.

**주요 처리 과정**

1. `PromptTemplate`을 사용하여 Supervisor의 시스템 프롬프트를 정의합니다.
2. `llm.with_structured_output(Task)`를 사용하여 LLM의 출력을 Pydantic `Task` 객체로 구조화합니다.
3. 기존 대화 메시지와 `get_outline()`으로 읽어 온 목차를 입력 데이터로 준비합니다.
4. `supervisor_chain.invoke()`를 호출하여 다음 작업과 담당 에이전트를 결정합니다.
5. 생성된 `Task`를 `task_history`에 추가합니다.
6. Supervisor의 작업 결정 내용을 `AIMessage`로 생성하여 `messages`에 추가합니다.
7. 변경된 상태 데이터를 반환합니다.

**참고**

- Supervisor는 직접 웹 검색이나 목차 작성을 수행하지 않고 다음 작업을 결정합니다.
- `Task.agent`는 `content_strategist`, `communicator`, `web_search_agent`, `vector_search_agent` 중 하나입니다.
- 생성된 `Task` 객체가 반드시 미완료 상태라는 별도의 검증 로직은 없습니다.

In [5]:
from langchain_core.messages import AIMessage
from langchain_core.prompts import PromptTemplate
#from utils import get_outline 
# #%run ./utils.ipynb로 불러온 get_outline 사용

def supervisor(state: State): # supervisor 에이전트 추가
    print("\n\n============ SUPERVISOR ============")

    # 시스템 프롬프트 정의
    supervisor_system_prompt = PromptTemplate.from_template(
        """
        너는 AI 팀의 supervisor로서 AI 팀의 작업을 관리하고 지도한다.
        사용자가 원하는 책을 써야 한다는 최종 목표를 염두에 두고, 
        사용자의 요구를 달성하기 위해 현재 해야할 일이 무엇인지 결정한다.

        supervisor가 활용할 수 있는 agent는 다음과 같다.     
        - content_strategist: 사용자의 요구사항이 명확해졌을 때 사용한다. AI 팀의 콘텐츠 전략을 결정하고, 전체 책의 목차(outline)를 작성한다. 
        - communicator: AI 팀에서 해야 할 일을 스스로 판단할 수 없을 때 사용한다. 사용자에게 진행상황을 사용자에게 보고하고, 다음 지시를 물어본다. 
        - web_search_agent: 웹 검색을 통해 목차(outline) 작성에 필요한 정보를 확보한다.
        - vector_search_agent: 벡터 DB 검색을 통해 목차(outline) 작성에 필요한 정보를 확보한다.

        아래 내용을 고려하여, 현재 해야할 일이 무엇인지, 사용할 수 있는 agent를 단답으로 말하라.

        ------------------------------------------
        previous_outline: {outline}
        ------------------------------------------
        messages:
        {messages}
        """
    )

    # 체인 연결
    supervisor_chain = supervisor_system_prompt | llm. with_structured_output(Task)    

    # 메시지 가져오기
    messages = state.get("messages", [])		#⑤

    # inputs 설정
    inputs = {
        "messages": messages,
        "outline": get_outline(current_path)
    }

    # task 문자열로 생성
    task = supervisor_chain.invoke(inputs) 	#⑦
    task_history = state.get("task_history", [])    # 작업 이력 가져오기
    task_history.append(task)                    	# 작업 이력에 추가

   
    # 메시지 추가
    supervisor_message = AIMessage(f"[Supervisor] {task}")
    messages.append(supervisor_message)
    print(supervisor_message.content)

    # state 업데이트
    return {
        "messages": messages, 
        "task_history": task_history
    }

## supervisor_router() — 에이전트 실행 경로 결정
Supervisor가 생성한 가장 최근 작업을 확인하고, 해당 작업을 수행할 에이전트의 이름을 반환하는 함수입니다.  
LangGraph의 조건부 분기(Conditional Edge)에서 다음 실행 노드를 선택하는 데 사용됩니다.

**매개변수**

| 매개변수 | 타입 | 설명 |
|---|---|---|
| `state` | `State` | 작업 이력이 포함된 현재 실행 상태 |

**State 주요 필드**

| 필드 | 설명 |
|---|---|
| `task_history` | Supervisor가 생성한 작업 이력 |
| `task_history[-1]` | 가장 최근에 생성된 작업 |
| `task_history[-1].agent` | 해당 작업을 담당할 에이전트 |

**반환값**

- `str`: 다음에 실행할 에이전트 이름을 반환합니다.
- 반환 가능한 값은 `Task.agent`에 정의된 네 가지 에이전트 이름입니다.

**주요 처리 과정**

1. `state["task_history"]`에서 작업 이력을 가져옵니다.
2. `[-1]` 인덱스를 사용하여 가장 최근에 생성된 `Task` 객체를 선택합니다.
3. 해당 작업의 `agent` 값을 반환합니다.
4. LangGraph는 반환된 이름을 기준으로 다음 실행 노드를 선택합니다.

In [ ]:
def supervisor_router(state: State):
    task = state['task_history'][-1]
    return task.agent			


## vector_search_agent() — 벡터 데이터베이스 검색 및 참고 자료 수집

벡터 데이터베이스에서 책의 목차 작성에 필요한 문서를 검색하는 함수입니다.  
LLM을 통해 검색 쿼리를 생성하고, 검색 결과를 수집하여 중복 문서를 제거한 뒤 공유 상태의 `references`에 저장합니다.

**매개변수**

| 매개변수 | 타입 | 설명 |
|---|---|---|
| `state` | `State` | 작업 이력, 대화 메시지, 참고 자료 등이 포함된 실행 상태 |

**State 주요 필드**

| 필드 | 설명 |
|---|---|
| `messages` | 기존 대화 및 에이전트 실행 기록 |
| `task_history` | 작업 이력 및 현재 수행할 작업 정보 |
| `references` | 기존 검색 쿼리와 참고 문서 |
| `references["queries"]` | 벡터 검색에 사용한 쿼리 목록 |
| `references["docs"]` | 검색된 문서 객체 목록 |

**반환값**

- `dict`: 갱신된 `messages`, `task_history`, `references`를 반환합니다.
- 검색 결과가 포함된 공유 상태를 LangGraph에 반영합니다.

**주요 처리 과정**

1. `task_history`에서 최근 작업을 가져오고, 담당 에이전트가 `vector_search_agent`인지 확인합니다.
2. 작업 설명(`task.description`), 기존 검색 기록, 대화 내용, 목차를 활용하여 검색 프롬프트를 생성합니다.
3. `llm.bind_tools([retrieve])`를 사용하여 LLM에 벡터 검색 도구를 연결합니다.
4. LLM을 호출하여 검색에 필요한 Tool Call을 생성합니다.
5. 각 Tool Call에 대해 `retrieve()`를 실행하여 관련 문서를 검색합니다.
6. 검색 쿼리와 문서 객체를 `references`에 추가합니다.
7. 문서의 `page_content`를 기준으로 중복된 검색 결과를 제거합니다.
8. 현재 작업의 `done`을 `True`로 변경하고 `done_at`에 완료 시간을 기록합니다.
9. 다음 작업을 담당할 `communicator`의 `Task` 객체를 생성합니다.
10. 검색 완료 메시지를 추가하고 갱신된 상태를 반환합니다.


In [7]:
# from tools import retrieve 
# #%run ./tools.ipynb로 불러온 retrieve 사용
from datetime import datetime

def vector_search_agent(state: State):
    print("\n\n============ VECTOR SEARCH AGENT ============")
    
    tasks = state.get("task_history", [])
    task = tasks[-1]
    if task.agent != "vector_search_agent":
        raise ValueError(f"Vector Search Agent가 아닌 agent가 Vector Search Agent를 시도하고 있습니다.\n {task}")

    vector_search_system_prompt = PromptTemplate.from_template(
        """
        너는 다른 AI Agent 들이 수행한 작업을 바탕으로, 
        목차(outline) 작성에 필요한 정보를 벡터 검색을 통해 찾아내는 Agent이다.

        현재 목차(outline)을 작성하는데 필요한 정보를 확보하기 위해, 
        다음 내용을 활용해 적절한 벡터 검색을 수행하라. 

        - 검색 목적: {mission}
        --------------------------------
        - 과거 검색 내용: {references}
        --------------------------------
        - 이전 대화 내용: {messages}
        --------------------------------
        - 목차(outline): {outline}
        """
    )

    # inputs 설정
    mission = task.description
    references = state.get("references", {"queries": [], "docs": []})
    messages = state["messages"]
    outline = get_outline(current_path)

    inputs = {
        "mission": mission,
        "references": references,
        "messages": messages,
        "outline": outline
    }

    # LLM과 벡터 검색 모델 연결
    llm_with_retriever = llm.bind_tools([retrieve]) 
    vector_search_chain = vector_search_system_prompt | llm_with_retriever

    # LLM과 벡터 검색 모델 연결
    search_plans = vector_search_chain.invoke(inputs)
    for tool_call in search_plans.tool_calls:
        print('-----------------------------------', tool_call)
        args = tool_call["args"]
        query = args["query"] 

        retrieved_docs = retrieve.invoke(args)

        references["queries"].append(query) 
        references["docs"] += retrieved_docs
    
    unique_docs = []
    unique_page_contents = set()

    for doc in references["docs"]:
        if doc.page_content not in unique_page_contents:
            unique_docs.append(doc)
            unique_page_contents.add(doc.page_content)

    references["docs"] = unique_docs

    # 검색 결과 출력 – 쿼리 출력
    print('Queries:--------------------------')
    queries = references["queries"]
    for query in queries:
        print(query)
    
    # 검색 결과 출력 – 문서 청크 출력
    print('References:--------------------------')
    for doc in references["docs"]:
        print(doc.page_content[:100])
        print('--------------------------')

    # task 완료
    tasks[-1].done = True
    tasks[-1].done_at = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

    # 새로운 task 추가
    new_task = Task(
        agent="communicator",
        done=False,
        description="AI팀의 진행상황을 사용자에게 보고하고, 사용자의 의견을 파악하기 위한 대화를 나눈다",
        done_at=""
    )
    tasks.append(new_task)

    # vector search agent의 작업후기를 메시지로 생성
    msg_str = f"[VECTOR SEARCH AGENT] 다음 질문에 대한 검색 완료: {queries}"
    message = AIMessage(msg_str)
    print(msg_str)

    messages.append(message)
    # state 업데이트
    return {
        "messages": messages,
        "task_history": tasks,
        "references": references
    }

## web_search_agent() — 웹 검색 및 Chroma DB 데이터 수집

웹 검색을 통해 책의 목차 작성에 필요한 외부 자료를 수집하는 함수입니다.  
LLM이 생성한 검색 쿼리를 바탕으로 웹 검색을 수행하고, 수집된 데이터를 Chroma 벡터 데이터베이스에 추가합니다.

**매개변수**

| 매개변수 | 타입 | 설명 |
|---|---|---|
| `state` | `State` | 현재 작업, 대화 메시지, 기존 참고 자료를 포함하는 실행 상태 |

**State 주요 필드**

| 필드 | 설명 |
|---|---|
| `messages` | 기존 대화 및 에이전트 실행 메시지 |
| `task_history` | 현재 웹 검색 작업 및 이전 작업 이력 |
| `references` | 기존 검색 기록과 참고 자료 |

**반환값**

- `dict`: 갱신된 `messages`와 `task_history`를 반환합니다.
- 웹 검색 결과는 별도 함수를 통해 Chroma DB에 추가됩니다.

**주요 처리 과정**

1. 가장 최근 작업을 가져와 담당 에이전트가 `web_search_agent`인지 확인합니다.
2. 작업 설명, 기존 검색 기록, 대화 내용, 목차, 현재 시간을 포함한 시스템 프롬프트를 구성합니다.
3. `llm.bind_tools([web_search])`를 통해 LLM에 웹 검색 도구를 연결합니다.
4. `web_search_chain.invoke()`를 호출하여 검색 Tool Call을 생성합니다.
5. 각 Tool Call에 포함된 검색 쿼리를 `queries` 리스트에 저장합니다.
6. `web_search.invoke()`로 웹 검색을 실행하고 검색 결과가 저장된 JSON 파일 경로를 가져옵니다.
7. `add_web_pages_json_to_chroma()`를 호출하여 해당 JSON 데이터를 Chroma DB에 추가합니다.
8. 현재 작업을 완료 처리하고 완료 시간을 기록합니다.
9. 새로 수집한 자료를 벡터 검색할 수 있도록 `vector_search_agent`의 작업을 생성합니다.
10. 웹 검색 완료 메시지를 `messages`에 추가하고 갱신된 상태를 반환합니다.

In [8]:
#from tools import web_search, add_web_pages_json_to_chroma 
# #%run ./tools.ipynb로 불러온 web_search, add_web_pages_json_to_chroma 사용

def web_search_agent(state: State): #① (0)
    print("\n\n============ WEB SEARCH AGENT ============")

    # 작업 리스트 가져와서 web search agent 가 할 일인지 확인하기
    tasks = state.get("task_history", [])
    task = tasks[-1]

    if task.agent != "web_search_agent":
        raise ValueError(f"Web Search Agent가 아닌 agent가 Web Search Agent를 시도하고 있습니다.\n {task}")
    
    #③ 시스템 프롬프트 정의
    web_search_system_prompt = PromptTemplate.from_template(
        """
        너는 다른 AI Agent 들이 수행한 작업을 바탕으로, 
        목차(outline) 작성에 필요한 정보를 웹 검색을 통해 찾아내는 Web Search Agent이다.

        현재 부족한 정보를 검색하고, 복합적인 질문은 나눠서 검색하라.

        - 검색 목적: {mission}
        --------------------------------
        - 과거 검색 내용: {references}
        --------------------------------
        - 이전 대화 내용: {messages}
        --------------------------------
        - 목차(outline): {outline}
        --------------------------------
        - 현재 시각 : {current_time}
        """
    )
    
    #④ 기존 대화 내용 가져오기
    messages = state.get("messages", [])

    #⑤ 인풋 자료 준비하기
    inputs = {
        "mission": task.description,
        "references": state.get("references", {"queries": [], "docs": []}),
        "messages": messages,
        "outline": get_outline(current_path),
        "current_time": datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    }

    #⑥ LLM과 웹 검색 모델 연결
    llm_with_web_search = llm.bind_tools([web_search])

    #⑦ 시스템 프롬프트와 모델을 연결
    web_search_chain = web_search_system_prompt | llm_with_web_search

    #⑧ 웹 검색 tool_calls 가져오기
    search_plans = web_search_chain.invoke(inputs)

    #⑨ 어떤 내용을 검색했는지 담아두기
    queries = []

    #⑩ 검색 계획(tool_calls)에 따라 검색하기
    for tool_call in search_plans.tool_calls:
        print('-------- web search --------', tool_call)
        args = tool_call["args"]
        
        queries.append(args["query"])

        # (10)  검색 결과를 chroma에 추가
        _, json_path = web_search.invoke(args)
        print('json_path:', json_path)

        # (10)  JSON 파일을 chroma에 추가
        add_web_pages_json_to_chroma(json_path)

    #⑪ (11) task 완료
    tasks[-1].done = True
    tasks[-1].done_at = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

    #⑪ (11) 새로운 task 추가
    task_desc = "AI팀이 쓸 책의 세부 목차를 결정하기 위한 정보를 벡터 검색을 통해 찾아낸다."
    task_desc += f" 다음 항목이 새로 추가되었다\n: {queries}"
    
    new_task = Task(
        agent="vector_search_agent",
        done=False,
        description=task_desc,
        done_at=""
    )

    tasks.append(new_task)

    #⑫ (12) 작업 후기 메시지
    msg_str = f"[WEB SEARCH AGENT] 다음 질문에 대한 검색 완료: {queries}"
    messages.append(AIMessage(msg_str))

    #⑬ (13) state 업데이트
    return {
        "messages": messages,
        "task_history": tasks
    }

## content_strategist() — 책의 목차 작성 및 수정

사용자의 요구사항과 수집된 참고 자료를 분석하여 책의 목차를 작성하거나 수정하는 함수입니다.  
LLM이 생성한 목차를 Markdown 파일로 저장하고, 작업 완료 상태를 기록합니다.

**매개변수**

| 매개변수 | 타입 | 설명 |
|---|---|---|
| `state` | `State` | 대화 내용, 작업 이력, 참고 자료가 포함된 실행 상태 |

**State 주요 필드**

| 필드 | 설명 |
|---|---|
| `messages` | 사용자와 AI 에이전트의 대화 내용 |
| `task_history` | 수행 중인 작업 및 이전 작업 이력 |
| `references` | 목차 작성에 활용할 참고 자료 |
| `references["queries"]` | 참고 자료 검색에 사용한 쿼리 |
| `references["docs"]` | 검색으로 확보한 참고 문서 |

**반환값**

- `dict`: 갱신된 `messages`와 `task_history`를 반환합니다.
- 저장 파일: `{current_path}/data/outline.md`

**주요 처리 과정**

1. 콘텐츠 전략가 역할을 수행할 시스템 프롬프트를 정의합니다.
2. `PromptTemplate`, LLM, `StrOutputParser`를 연결하여 목차 생성 체인을 구성합니다.
3. `get_outline()`을 호출하여 기존에 작성된 목차를 불러옵니다.
4. 기존 목차, 사용자 대화, 참고 자료를 입력 데이터로 준비합니다.
5. `content_strategist_chain.stream()`을 사용하여 목차를 스트리밍 방식으로 생성합니다.
6. 생성된 텍스트를 `gathered` 문자열에 누적합니다.
7. `save_outline()`을 호출하여 목차를 Markdown 파일로 저장합니다.
8. 목차 작성 완료 메시지를 `messages`에 추가합니다.
9. 현재 작업이 `content_strategist`의 작업인지 확인하고 완료 상태와 시간을 기록합니다.
10. 다음 작업인 `communicator`의 `Task`를 생성하고 상태를 반환합니다.

In [9]:
from langchain_core.output_parsers.string import StrOutputParser
# from utils import save_outline
# #%run ./utils.ipynb로 불러온 save_outline 사용

# 목차를 작성하는 노드(agent)
def content_strategist(state: State):
    print("\n\n============ CONTENT STRATEGIST ============")

    # 시스템 프롬프트 정의
    content_strategist_system_prompt = PromptTemplate.from_template(
        """
        너는 책을 쓰는 AI팀의 콘텐츠 전략가(Content Strategist)로서,
        이전 대화 내용을 바탕으로 사용자의 요구사항을 분석하고, AI팀이 쓸 책의 세부 목차를 결정한다.

        지난 목차가 있다면 그 버전을 사용자의 요구에 맞게 수정하고, 없다면 새로운 목차를 제안한다.
        목차를 작성하는데 필요한 정보는 "참고 자료"에 있으므로 활용한다. 

        --------------------------------
        - 지난 목차: {outline}
        --------------------------------
        - 이전 대화 내용: {messages}
        --------------------------------
        - 참고 자료: {references}
        """
    )

    # 시스템 프롬프트와 모델을 연결
    content_strategist_chain = content_strategist_system_prompt | llm | StrOutputParser()

    messages = state["messages"]        # 상태에서 메시지를 가져옴
    outline = get_outline(current_path) # 저장된 목차를 가져옴

    # 입력값 정의
    inputs = {
        "messages": messages,
        "outline": outline, 
        "references": state.get("references", {"queries": [], "docs": []})
    }

    # 목차 작성
    gathered = ''
    for chunk in content_strategist_chain.stream(inputs):
        gathered += chunk
        print(chunk, end='')

    print()

    save_outline(current_path, gathered) # 목차 저장

    # 메시지 추가    
    content_strategist_message = f"[Content Strategist] 목차 작성 완료"
    print(content_strategist_message)
    messages.append(AIMessage(content_strategist_message))

    task_history = state.get("task_history", []) # task_history 가져오기
    # 최근 task 작업완료(done) 처리하기
    if task_history[-1].agent != "content_strategist": 
        raise ValueError(f"Content Strategist가 아닌 agent가 목차 작성을 시도하고 있습니다.\n {task_history[-1]}")
    
    task_history[-1].done = True
    task_history[-1].done_at = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

    # 다음 작업이 communicator로 사용자와 대화하는 것이므로 새 작업 추가 
    new_task = Task(
        agent="communicator",
        done=False,
        description="AI팀의 진행상황을 사용자에게 보고하고, 사용자의 의견을 파악하기 위한 대화를 나눈다",
        done_at=""
    )
    task_history.append(new_task)

    print(new_task)

    # 현재 state를 업데이트한다. 
    return {
        "messages": messages,
        "task_history": task_history
    }

## communicator() — 사용자 커뮤니케이션 및 진행 상황 보고

AI 에이전트 팀의 작업 진행 상황을 사용자에게 보고하고, 사용자와 대화를 수행하는 함수입니다.  
기존 대화와 작성된 목차를 참고하여 응답을 생성하고, 이를 공유 상태에 기록합니다.

**매개변수**

| 매개변수 | 타입 | 설명 |
|---|---|---|
| `state` | `State` | 대화 메시지 및 작업 이력이 포함된 실행 상태 |

**State 주요 필드**

| 필드 | 설명 |
|---|---|
| `messages` | 사용자와 에이전트 간의 대화 기록 |
| `task_history` | 현재 커뮤니케이션 작업 및 이전 작업 이력 |

**반환값**

- `dict`: 갱신된 `messages`와 `task_history`를 반환합니다.
- 생성된 사용자 응답은 콘솔에도 출력됩니다.

**주요 처리 과정**

1. 사용자와 대화하는 커뮤니케이터 역할의 시스템 프롬프트를 정의합니다.
2. `PromptTemplate`과 LLM을 연결하여 응답 생성 체인을 구성합니다.
3. 기존 대화 메시지와 `get_outline()`으로 읽은 목차를 입력 데이터로 준비합니다.
4. `system_chain.stream()`을 통해 LLM 응답을 스트리밍 방식으로 생성합니다.
5. 생성되는 응답을 콘솔에 출력하면서 `gathered`에 누적합니다.
6. 완성된 응답 메시지를 `messages`에 추가합니다.
7. 현재 작업이 `communicator`의 작업인지 확인합니다.
8. 해당 작업을 완료 처리하고 완료 시간을 기록합니다.
9. 갱신된 상태 데이터를 반환합니다.

**참고**
- 기존 목차는 LLM이 응답을 생성할 때 참고 자료로 사용합니다.
- 시스템 프롬프트에서 목차를 다시 출력하지 않도록 지시합니다.

In [10]:

def communicator(state: State):
    print("\n\n============ COMMUNICATOR ============")

    # 시스템 프롬프트 정의
    communicator_system_prompt = PromptTemplate.from_template(
        """
        너는 책을 쓰는 AI팀의 커뮤니케이터로서, 
        AI팀의 진행상황을 사용자에게 보고하고, 사용자의 의견을 파악하기 위한 대화를 나눈다. 

        사용자도 outline(목차)을 이미 보고 있으므로, 다시 출력할 필요는 없다.
        outline: {outline} 
        --------------------------------
        messages: {messages}
        """
    )

    #② 시스템 프롬프트와 모델을 연결
    system_chain = communicator_system_prompt | llm

    # 상태에서 메시지를 가져옴
    messages = state["messages"]

    # 입력값 정의
    inputs = {
        "messages": messages,
        "outline": get_outline(current_path)
    }

    # 스트림되는 메시지를 출력하면서, gathered에 모으기
    gathered = None

    print('\nAI\t: ', end='')
    for chunk in system_chain.stream(inputs):
        print(chunk.content, end='')

        if gathered is None:
            gathered = chunk
        else:
            gathered += chunk

    messages.append(gathered)

    task_history = state.get("task_history", []) 
    if task_history[-1].agent != "communicator":
        raise ValueError(f"Communicator가 아닌 agent가 대화를 시도하고 있습니다.\n {task_history[-1]}")
    
    task_history[-1].done = True
    task_history[-1].done_at = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

    return {
        "messages": messages,
        "task_history": task_history
    }


In [11]:
from langgraph.graph import StateGraph, START, END

# 상태 그래프 정의
graph_builder = StateGraph(State)

# Nodes
graph_builder.add_node("supervisor", supervisor)
graph_builder.add_node("communicator", communicator)
graph_builder.add_node("content_strategist", content_strategist)
graph_builder.add_node("vector_search_agent", vector_search_agent)
graph_builder.add_node("web_search_agent", web_search_agent)

# Edges
graph_builder.add_edge(START, "supervisor")
graph_builder.add_conditional_edges(
    "supervisor",
    supervisor_router,
    {
        "content_strategist": "content_strategist",
        "communicator": "communicator",
        "vector_search_agent": "vector_search_agent",
        "web_search_agent": "web_search_agent"
    }
)
graph_builder.add_edge("content_strategist", "communicator")
graph_builder.add_edge("web_search_agent", "vector_search_agent") #③
graph_builder.add_edge("vector_search_agent", "communicator")
graph_builder.add_edge("communicator", END)

graph = graph_builder.compile()

In [12]:
# graph.get_graph().draw_mermaid_png(output_file_path=absolute_path.replace('.ipynb', '.png'))

In [13]:
from langchain_core.messages import SystemMessage

# 상태 초기화
state = State(
    messages = [
        SystemMessage(
                f"""
            너희 AI들은 사용자의 요구에 맞는 책을 쓰는 작가팀이다.
            사용자가 사용하는 언어로 대화하라.

            현재시각은 {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}이다.

            """
        )
    ],
    task_history=[]
)

In [ ]:
from langchain_core.messages import HumanMessage
# from utils import save_state
# %run ./utils.ipynb로 불러온 save_state 사용

while True:
    user_input = input("\nUser\t: ").strip()

    if user_input.lower() in ['exit', 'quit', 'q']:
        print("Goodbye!")
        break
    
    state["messages"].append(HumanMessage(user_input))
    state = graph.invoke(state)

    print('\n------------------------------------ MESSAGE COUNT\t', len(state["messages"]))

    save_state(current_path, state) # 현재 state 내용 저장
